# Platinum performance (val only)

In [1]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
pio.renderers.default = "plotly_mimetype"

HERE = Path.cwd().resolve()
REPO = None
for p in [HERE, *HERE.parents]:
    if (p / "platinum" / "results" / "leaderboard.csv").exists():
        REPO = p
        break
if REPO is None:
    raise FileNotFoundError("platinum/results/leaderboard.csv not found")

RESULTS = REPO / "platinum" / "results"
FIG = RESULTS / "figures"
FIG.mkdir(parents=True, exist_ok=True)
GOLD = REPO / "data" / "gold" / "delay"

lb = pd.read_csv(RESULTS / "leaderboard.csv") if (RESULTS / "leaderboard.csv").exists() else pd.DataFrame()
audit = {}
ap = GOLD / "label_audit.json"
if ap.exists():
    audit = json.loads(ap.read_text(encoding="utf-8"))
display(Markdown("# Platinum — from approved MW to deliverable MW"))
display(Markdown(
    "Val-only COD-slip models. We are **not** claiming a MISO Step-Up failure record; "
    "the process is still proposed. This report measures generator COD execution risk "
    "that a future Firm Service Step-Up increment would revalidate."
))
display(Markdown(
    f"Follow-up window **{audit.get('followup_months', 24)} months**. "
    f"Val labeled **{audit.get('val_labeled')}**. "
    f"Neural nets allowed: **{audit.get('neural_nets_ok')}**."
))
display(lb)

# Platinum — from approved MW to deliverable MW

Val-only COD-slip models. We are **not** claiming a MISO Step-Up failure record; the process is still proposed. This report measures generator COD execution risk that a future Firm Service Step-Up increment would revalidate.

Follow-up window **24 months**. Val labeled **1054**. Neural nets allowed: **True**.

,model,run_name,split,status,mae,rmse,median_ae,pinball80,companion_pr_auc,delayed_mw_capture_at_10pct,beats_persist_mae,n_features,reason
0,catboost,catboost,val,ok,12.039561,21.062791,7.317897e-07,9.357125,0.551975,0.193755,True,62.0,NaN
1,xgboost,xgboost,val,ok,12.053402,21.086875,1.358874e-03,9.370279,0.347632,0.113088,False,92.0,NaN
2,seq_cnn,seq_cnn,val,ok,12.084156,21.011121,1.902420e-01,9.343199,0.404815,0.149144,False,5.0,NaN
3,ft_transformer,ft_transformer,val,ok,12.084663,21.059451,1.784954e-01,9.366877,0.350965,0.079264,False,62.0,NaN
4,tabm,tabm,val,ok,12.095970,20.989133,2.341000e-01,9.333543,0.282695,0.051350,False,62.0,NaN
5,lightgbm,lightgbm,val,ok,12.118608,20.896059,3.647166e-01,9.293493,0.488420,0.158985,False,92.0,NaN
6,nasnet_cnn,nasnet_cnn,val,ok,12.185008,20.770639,8.489167e-01,9.249699,0.369684,0.108581,False,92.0,NaN
7,logistic,logistic,val,ok,12.341106,21.141951,7.782365e-01,9.493320,0.325836,0.112526,False,92.0,NaN
8,survival,survival,val,ok,21.072993,33.153247,1.536857e+01,10.922320,0.632288,0.110916,False,66.0,NaN
9,timesfm,timesfm,val,ok,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## GIA delay distribution

In [2]:
panel_path = GOLD / "cod_delay_panel.parquet"
if panel_path.exists():
    panel = pd.read_parquet(panel_path)
    lab = panel[panel["cod_slip_months_next_12m"].notna() & panel["split"].eq("val")].copy()
    if "is_gia" in lab.columns:
        gia = lab[lab["is_gia"].fillna(False)]
    else:
        gia = lab
    slip = pd.to_numeric(gia["cod_slip_months_next_12m"], errors="coerce") if len(gia) else pd.Series(dtype=float)
    mw = pd.to_numeric(gia.get("capacity_mw"), errors="coerce").fillna(0) if len(gia) else pd.Series(dtype=float)
    delayed = slip >= 12
    display(Markdown("## GIA / advanced-study slice (slide 4)"))
    display(Markdown(
        f"Val GIA rows **{len(gia)}**. Median slip **{float(slip.median()) if len(slip) else float('nan'):.1f}** months. "
        f"Share ≥12m **{(delayed.mean() if len(slip) else float('nan')):.1%}**. "
        f"Delayed GIA MW **{float(mw[delayed].sum()) if len(gia) else 0:.0f}**."
    ))
    fig = go.Figure(go.Histogram(x=slip, nbinsx=20, name="val GIA slip"))
    fig.update_layout(title="Val COD slip months — GIA/advanced subset", xaxis_title="months")
    fig.write_html(FIG / "gia_slip.html")
    try:
        fig.write_image(FIG / "gia_slip.png")
    except Exception as e:
        print("png skipped:", e)
    try:
        fig.show()
    except Exception as e:
        print("show skipped:", e)
else:
    display(Markdown("_cod_delay_panel.parquet missing — run scripts/run_delay_gold.py_"))

## GIA / advanced-study slice (slide 4)

Val GIA rows **284**. Median slip **23.0** months. Share ≥12m **72.5%**. Delayed GIA MW **33510**.

png skipped: 

Kaleido requires Google Chrome to be installed.

Either download and install Chrome yourself following Google's instructions for your operating system,
or install it from your terminal by running:

    $ plotly_get_chrome




## Leaderboard

In [3]:
ok = lb[lb["status"].isin(["ok", "not_competitive"])].copy() if len(lb) else lb
if len(ok) and "mae" in ok.columns:
    fig = go.Figure(go.Bar(x=ok["model"], y=ok["mae"], name="val MAE"))
    fig.update_layout(title="Val MAE (lower is better). Test sealed.", yaxis_title="months")
    fig.write_html(FIG / "mae_leaderboard.html")
    try:
        fig.write_image(FIG / "mae_leaderboard.png")
    except Exception as e:
        print("png skipped:", e)
    try:
        fig.show()
    except Exception as e:
        print("show skipped:", e)
    if "companion_pr_auc" in ok.columns:
        fig2 = go.Figure(go.Bar(x=ok["model"], y=ok["companion_pr_auc"], name="companion PR-AUC"))
        fig2.update_layout(title="Companion P(slip ≥ 12m) PR-AUC")
        fig2.write_html(FIG / "companion_pr.html")
        try:
            fig2.show()
        except Exception as e:
            print("show skipped:", e)

png skipped: 

Kaleido requires Google Chrome to be installed.

Either download and install Chrome yourself following Google's instructions for your operating system,
or install it from your terminal by running:

    $ plotly_get_chrome




In [4]:
display(Markdown("## Learning curves"))
for model in ["catboost", "xgboost", "lightgbm", "seq_cnn", "nasnet_cnn", "ft_transformer", "tabm"]:
    p = RESULTS / model / "history.csv"
    if not p.exists():
        continue
    h = pd.read_csv(p)
    if h.empty:
        continue
    fig = go.Figure()
    for col in ("train_loss", "val_loss", "val_mae", "train_mae"):
        if col in h.columns:
            fig.add_trace(go.Scatter(x=h.get("step"), y=h[col], mode="lines", name=col))
    fig.update_layout(title=f"{model} history")
    fig.write_html(FIG / f"history_{model}.html")
    try:
        fig.show()
    except Exception as e:
        print("show skipped:", e)

## Learning curves

In [5]:
display(Markdown("## What this is not"))
display(Markdown(
    "- Not a DPP restudy. Not another queue. Not a second Resource Adequacy test.\\n"
    "- FERC-730 was **not** a clean bulk table; transmission delay is a NERC/MTEP callout "
    "(110 of 1,160 NERC LTRA projects delayed), not a Platinum model.\\n"
    "- NASNet-as-image is included to **actually train** (100+20), not because it should win."
))
display(Markdown("Study the ramp once. Revalidate only what changes."))

## What this is not

- Not a DPP restudy. Not another queue. Not a second Resource Adequacy test.\n- FERC-730 was **not** a clean bulk table; transmission delay is a NERC/MTEP callout (110 of 1,160 NERC LTRA projects delayed), not a Platinum model.\n- NASNet-as-image is included to **actually train** (100+20), not because it should win.

Study the ramp once. Revalidate only what changes.